# Viora — Free GPU Training (Colab / Kaggle)

Train Viora's **own** video-language model for **$0** on a free **T4 (16 GB)** — plenty for the
SigLIP + Qwen-0.5B + LoRA ("pragmatic") model. Free sessions time out (4–12 h), so this notebook
**checkpoints** to Drive/output and can **resume** across sessions.

**Before running — pick a T4, not a P100:**
- **Kaggle:** right panel → *Session options* → Accelerator → **GPU T4 x2** (30 free GPU-hrs/week)
- **Colab:** Runtime → Change runtime type → **T4 GPU**

> ⚠️ On Kaggle, do **NOT** pick **GPU P100**. The P100 is compute capability `sm_60`, which current
> PyTorch dropped support for — it fails with *"CUDA error: no kernel image is available"*. The T4 is
> `sm_75` and fully supported. Cell 1 checks this and stops early with instructions if you're on a P100.

This is Viora's own model — no wrapper, no external answering API.

In [ ]:
# 1) Confirm the GPU is compatible with the installed torch (the #1 free-GPU gotcha).
import torch
assert torch.cuda.is_available(), "No GPU! Turn on the Accelerator (Kaggle Session options / Colab Runtime)."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | GPU: {name}  (compute capability sm_{major}{minor})")

# Modern PyTorch supports sm_70+ only. Kaggle's older 'GPU P100' is sm_60 -> incompatible.
if (major, minor) < (7, 0):
    raise SystemExit(
        f"\n>>> {name} is compute capability {major}.{minor} (sm_{major}{minor}), which current "
        "PyTorch does NOT support (needs sm_70+).\n"
        ">>> FIX (one setting): switch the accelerator to a T4 (sm_75):\n"
        "      Kaggle: right panel -> 'Session options' -> Accelerator -> 'GPU T4 x2'   (NOT 'GPU P100')\n"
        "      Colab:  Runtime -> Change runtime type -> 'T4 GPU'\n"
        ">>> Then re-run from cell 1. The T4 is free and fully supported; the P100 is not.\n"
    )

# Backstop: actually run a CUDA kernel (catches any other torch/GPU mismatch).
try:
    _ = (torch.randn(8, 8, device="cuda") @ torch.randn(8, 8, device="cuda")).sum().item()
    print("CUDA kernel test: OK ✓  (torch matches this GPU)")
except Exception as e:
    raise SystemExit(
        f"torch cannot run a kernel on this GPU: {e}\n"
        ">>> Switch the accelerator to 'GPU T4 x2' (Kaggle) / 'T4 GPU' (Colab) and re-run."
    )

In [ ]:
# 2) Get the Viora code from GitHub (absolute paths; re-run pulls the latest fixes)
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # fail fast instead of hanging on a login prompt

REPO_URL = "https://github.com/garvbahl37-gif/Viora.git"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(BASE, "viora")

# If the repo is PRIVATE: add a GitHub token in Kaggle (Add-ons -> Secrets) as
# GITHUB_TOKEN, then uncomment the next two lines:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@github.com/garvbahl37-gif/Viora.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    proc = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, capture_output=True, text=True)
else:
    proc = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)

if proc.returncode != 0:
    print(proc.stdout, proc.stderr)
    _out = proc.stdout + proc.stderr
    if "No space left on device" in _out:
        # This is a DISK-FULL error, not a repo-access/network problem -- making the repo
        # public or enabling Internet will NOT fix it (a common false lead: the message
        # below never applies here). Free space first, THEN re-run this cell.
        raise SystemExit(
            "Git failed: the disk is FULL. This is NOT an access/network issue -- ignore\n"
            "any 'make repo public' suggestion for this specific error. Free space first:\n"
            "  !rm -f /kaggle/working/pragmatic/*.tmp\n"
            "  !rm -f /kaggle/working/viora/data/shards/*.tar\n"
            "  !rm -rf ~/.cache/huggingface/hub\n"
            "then re-run this cell. If it STILL fails after that, the clone itself may be\n"
            "corrupted from the disk-full write -- delete and let this cell re-clone fresh:\n"
            "  !rm -rf /kaggle/working/viora"
        )
    raise SystemExit(
        "Git failed. Fix one of these:\n"
        "  1) Make the repo PUBLIC (simplest), or set GITHUB_TOKEN above for a private repo.\n"
        "  2) Enable Internet: Kaggle right panel -> Internet -> On (phone-verify your account)."
    )
%cd {REPO_DIR}
print("working dir:", os.getcwd())

In [ ]:
# 3) Install Viora WITHOUT reinstalling torch.
#    Kaggle/Colab ship a CUDA-matched torch; letting pip pull torch from PyPI causes
#    "CUDA error: no kernel image is available for execution on the device".
#    So: install the package with --no-deps, then install only the extra deps explicitly.
import torch
print("keeping torch", torch.__version__, "| cuda", torch.cuda.is_available())
!pip install -q -e . --no-deps
!pip install -q einops omegaconf pyyaml tqdm rich av webdataset peft \
    transformers safetensors huggingface_hub
# Kaggle/Colab ship an old torchao (0.10) that breaks peft's LoRA dispatch (also handled in code).
!pip uninstall -q -y torchao 2>/dev/null || true
!python scripts/validate_environment.py

In [ ]:
# 4) Choose an output dir that SURVIVES session end (so you can resume).
#    Colab -> Google Drive;  Kaggle -> /kaggle/working (downloadable).
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/viora_runs/pragmatic'
except Exception:
    OUT = '/kaggle/working/pragmatic' if os.path.isdir('/kaggle') else 'runs/pragmatic'
os.makedirs(OUT, exist_ok=True)
print('checkpoints ->', OUT)

In [ ]:
# 4b) Resume from HF if a checkpoint was pushed in a PREVIOUS session (else start from
#      the last local one, else start fresh). Set HF_REPO to your model repo.
HF_REPO = "bharatverse11/viora-msrvtt"
HF_TOKEN = None  # paste a HF token here if HF_REPO is PRIVATE; never commit a token to git

import torch
from viora.training.checkpointing import latest_checkpoint
from viora.utils.hf_relay import pull_checkpoint_from_hf

RESUME_FROM = ""
pulled = pull_checkpoint_from_hf(HF_REPO, "final.pt", local_dir=OUT, token=HF_TOKEN)
if pulled is not None:
    RESUME_FROM = str(pulled)
    print(f"resuming from HF checkpoint -> {RESUME_FROM}")
else:
    local = latest_checkpoint(OUT)   # most-recently-written local checkpoint, if any
    RESUME_FROM = str(local) if local else ""
    print(f"no HF checkpoint yet -> {'resuming locally from ' + RESUME_FROM if RESUME_FROM else 'starting fresh'}")
    if HF_TOKEN is None:
        # HF reports a PRIVATE repo as "not found" to an unauthenticated request (by design,
        # so it doesn't leak whether the repo exists) -- so this message is ambiguous between
        # "genuinely nothing pushed yet" and "it exists but you're not authorized to see it".
        print("   NOTE: if you already pushed a checkpoint to this repo and it's private, "
              "set HF_TOKEN above and re-run this cell -- otherwise you may be about to "
              "retrain from scratch instead of resuming.")

# training.max_steps is an ABSOLUTE step count, and resuming restores the checkpoint's OWN
# step (it does NOT reset to 0) -- so cell 6 needs to know where we're resuming FROM to
# compute how many MORE steps to actually run this session (see cell 6).
RESUME_STEP = 0
if RESUME_FROM:
    RESUME_STEP = int(torch.load(RESUME_FROM, map_location="cpu", weights_only=False).get("step", 0))
    print(f"resumed checkpoint is at step {RESUME_STEP}")


In [ ]:
# 5) Data. Synthetic shards so the whole pipeline runs end-to-end for free.
#    Swap in a real dataset (MSR-VTT) later for a useful model — see docs/PRODUCTION.md.
!python scripts/build_shards.py --synthetic 3000 --out data/shards/train-%06d.tar
import glob
_n = len(glob.glob("data/shards/train-*.tar"))          # robust to the actual shard count
SHARDS = "data/shards/train-{000000..%06d}.tar" % (_n - 1)
print(f"{_n} shards -> {SHARDS}")

In [ ]:
# 5b) REAL MSR-VTT captions -> real shards. You have the videos; this fetches the captions.
#      Uses a local *videodatainfo.json* if you added one; otherwise downloads the captions
#      from HuggingFace (friedrichor/MSR-VTT -> msrvtt_train_7k.json = the 7,010 train clips).
import glob, os
INPUT_ROOT = "/kaggle/input"   # Colab: point this at your mounted dataset directory

mp4s = glob.glob(f"{INPUT_ROOT}/**/*.mp4", recursive=True)
if not mp4s:
    print(f"No .mp4 under {INPUT_ROOT} -> keeping synthetic shards from cell 5.")
    print("Add the MSR-VTT VIDEOS dataset (right panel -> Add Data) to train for real.")
else:
    parts = os.path.abspath(mp4s[0]).split(os.sep)
    VIDEOS_DIR = (os.sep.join(parts[: parts.index("input") + 2])
                  if "input" in parts else os.path.dirname(mp4s[0]))
    print(f"{len(mp4s)} videos under {VIDEOS_DIR}")

    # captions: prefer a local videodatainfo.json; else pull from HuggingFace (public, no token).
    local = ([p for p in glob.glob(f"{INPUT_ROOT}/**/*videodatainfo*.json", recursive=True)
              if "train" in os.path.basename(p).lower()]
             or glob.glob(f"{INPUT_ROOT}/**/*videodatainfo*.json", recursive=True))
    if local:
        ANNOT = local[0]
    else:
        from huggingface_hub import hf_hub_download
        print("no local captions -> downloading friedrichor/MSR-VTT captions from HuggingFace...")
        ANNOT = hf_hub_download("friedrichor/MSR-VTT", "msrvtt_train_7k.json", repo_type="dataset")
    print("captions:", ANNOT)

    # --format auto handles both a videodatainfo object AND a list of {video_id, caption}
    !python scripts/prepare_video_dataset.py \
        --videos "{VIDEOS_DIR}" --annotations "{ANNOT}" \
        --format auto --split train \
        --out data/shards/msrvtt-train-%06d.tar --maxcount 500
    _n = len(glob.glob("data/shards/msrvtt-train-*.tar"))
    if _n:
        SHARDS = "data/shards/msrvtt-train-{000000..%06d}.tar" % (_n - 1)
        print(f"OK -> training will use REAL shards: {SHARDS}")
    else:
        print("!! 0 shards written -- caption video_ids don't match your .mp4 filenames.")
        print("   your video files look like:", [os.path.basename(p) for p in mp4s[:3]])

In [ ]:
# 5c) COMBINED captions + MSRVTT-QA -> one shard set for BOTH tasks (multi-task training).
#      Reuses the SAME videos as cell 5b; only fetches the QA annotations. Also builds a
#      small held-out VAL split (captions' "validate" split, + val QA if available) so
#      training can log a real val_loss. Nothing here re-downloads any video.
#
#      QA source: the OFFICIAL MSRVTT-QA json published by Salesforce LAVIS (verified live),
#      with a couple of HF mirrors tried as a last-resort fallback in case that URL ever moves.
import glob, os

QA_URL_CANDIDATES = [
    "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/datasets/msrvtt/qa_train.json",
]
QA_VAL_URL_CANDIDATES = [
    "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/datasets/msrvtt/qa_val.json",
]
QA_HF_CANDIDATES = [("morpheushoc/msrvtt-qa", "train_qa.json"), ("morpheushoc/msrvtt-qa", "msrvtt_qa_train.json")]
QA_VAL_HF_CANDIDATES = [("morpheushoc/msrvtt-qa", "val_qa.json"), ("morpheushoc/msrvtt-qa", "msrvtt_qa_val.json")]

if "VIDEOS_DIR" not in globals():
    print("cell 5b did not find real videos -> skipping combined caption+QA build.")
    MIXED_SHARDS = None
    VAL_SHARDS = None
else:
    def _try_fetch(urls, hf_candidates, dest_name):
        import json as _json
        import urllib.request

        os.makedirs("data", exist_ok=True)
        dest = os.path.join("data", dest_name)
        for url in urls:
            try:
                urllib.request.urlretrieve(url, dest)
                with open(dest) as f:
                    _json.load(f)  # verify it's actually valid JSON before trusting it
                print(f"  found QA annotations: {url}")
                return dest
            except Exception as e:
                print(f"  {url} unavailable ({type(e).__name__}); trying next...")
        from huggingface_hub import hf_hub_download
        for repo, fname in hf_candidates:
            try:
                path = hf_hub_download(repo, fname, repo_type="dataset")
                print(f"  found QA annotations: {repo}/{fname}")
                return path
            except Exception as e:
                print(f"  {repo}/{fname} unavailable ({type(e).__name__}); trying next...")
        return None

    print("fetching MSRVTT-QA train annotations...")
    QA_TRAIN = _try_fetch(QA_URL_CANDIDATES, QA_HF_CANDIDATES, "msrvtt_qa_train.json")

    if QA_TRAIN is None:
        print("!! no MSRVTT-QA source worked -> training will stay caption-only (SHARDS from cell 5b).")
        MIXED_SHARDS = None
        VAL_SHARDS = None
    else:
        !python scripts/prepare_video_dataset.py \
            --videos "{VIDEOS_DIR}" --annotations "{ANNOT}" --qa-annotations "{QA_TRAIN}" \
            --format auto --split train \
            --out data/shards/msrvtt-mixed-%06d.tar --maxcount 500
        _n = len(glob.glob("data/shards/msrvtt-mixed-*.tar"))
        if _n:
            MIXED_SHARDS = "data/shards/msrvtt-mixed-{000000..%06d}.tar" % (_n - 1)
            SHARDS = MIXED_SHARDS   # cell 6 trains on the mixed set from here on
            print(f"OK -> training will use COMBINED caption+QA shards: {MIXED_SHARDS}")

            # cell 5b's caption-only shards store the SAME videos (now duplicated in the
            # combined set) -- they're pure waste once SHARDS points at MIXED_SHARDS, and on
            # a 20GB Kaggle disk this duplication is usually the difference between fitting
            # and a mid-write "disk full" crash. Safe to delete: nothing still reads them.
            _old_caption_only = glob.glob("data/shards/msrvtt-train-*.tar")
            if _old_caption_only:
                _freed = sum(os.path.getsize(p) for p in _old_caption_only) / 1e9
                for p in _old_caption_only:
                    os.remove(p)
                print(f"removed {len(_old_caption_only)} superseded caption-only shard(s), "
                      f"freed {_freed:.1f} GB")
        else:
            print("!! 0 combined shards written -> keeping caption-only SHARDS from cell 5b.")
            MIXED_SHARDS = None

        # held-out validation: only if the local videodatainfo.json actually has a "validate" split
        VAL_SHARDS = None
        try:
            QA_VAL = _try_fetch(QA_VAL_URL_CANDIDATES, QA_VAL_HF_CANDIDATES, "msrvtt_qa_val.json")
            _qa_val_flag = f'--qa-annotations "{QA_VAL}"' if QA_VAL else ""
            !python scripts/prepare_video_dataset.py \
                --videos "{VIDEOS_DIR}" --annotations "{ANNOT}" {_qa_val_flag} \
                --format auto --split validate \
                --out data/shards/msrvtt-val-%06d.tar --maxcount 500 --limit 300
            _vn = len(glob.glob("data/shards/msrvtt-val-*.tar"))
            if _vn:
                VAL_SHARDS = "data/shards/msrvtt-val-{000000..%06d}.tar" % (_vn - 1)
                print(f"val shards -> {VAL_SHARDS}")
        except Exception as e:
            print(f"no 'validate' split available ({type(e).__name__}) -> skipping val_loss logging.")

In [ ]:
# 6) Train: LoRA on Qwen-0.5B + frozen SigLIP + Viora's trainable bridge.
#    T4 supports fp16 (NOT bf16); small num_workers for Kaggle's limited CPUs.
#    Checkpoints to {OUT} every 200 steps -> resumable across free sessions.
#    (If you hit out-of-memory, lower training.batch_size to 2.)
#
#    training.max_steps is an ABSOLUTE target (resume restores the checkpoint's own step
#    count, it does NOT reset to 0) -- so this computes "where we are now" + "this
#    session's budget", capped at the overall goal, instead of a flat per-session number
#    that would do almost nothing once you're already resuming from a late checkpoint.
#
#    --qa-prob raised 0.5 -> 0.7 for this final session (weight QA more heavily, per
#    request to prioritize question-answering ability); still keeps some captioning
#    (0.3) so it doesn't fully forget how to describe. Also: hf_tokenize_fn (scripts/
#    train.py) now masks the "Question: ...Answer: " prefix out of the loss for QA
#    views -- only the answer (+EOS) drives the loss, instead of diluting the
#    gradient across the (given, ungenerated) question text too.
PER_SESSION_STEPS = 8000     # how much MORE to train this session (~5.3h at ~0.42 it/s)
TOTAL_TARGET_STEPS = 30000   # overall goal across all sessions

# NOTE on raising TOTAL_TARGET_STEPS: the cosine LR schedule is a pure function of
# (step, max_steps), and resuming only restores last_epoch -- so the target also sets
# the learning rate you resume AT. Finishing a run leaves lr parked at min_lr
# (1e-6 = effectively no learning), and resuming with the SAME target would keep it
# there. Measured at step 16400: target 16400 -> 1.0e-6 (dead), 20000 -> 9.1e-6,
# 30000 -> 4.4e-5 (healthy). Raise the target and the LR revives with it.

TARGET_STEPS_THIS_SESSION = min(RESUME_STEP + PER_SESSION_STEPS, TOTAL_TARGET_STEPS)
print(f"resumed at step {RESUME_STEP} -> training to step {TARGET_STEPS_THIS_SESSION} "
      f"(overall target {TOTAL_TARGET_STEPS})")

_resume_flag = f"training.resume={RESUME_FROM}" if RESUME_FROM else ""
# IMPORTANT: keep every "--flag value" OPTIONAL argument grouped together, and every
# bare "key=value" CONFIG OVERRIDE grouped together in ONE later block. train.py's
# `overrides` positional (nargs='*') only captures the FIRST contiguous run of
# non-flag tokens it meets; a bare override sandwiched between two "--flag" options
# (e.g. between --val-shards and --qa-prob) gets "used up" there, and the real
# override block that follows --qa-prob is then rejected as "unrecognized arguments".
_val_shards_flag = f'--val-shards "{VAL_SHARDS}"' if VAL_SHARDS else ""
_eval_every_override = "training.eval_every=500" if VAL_SHARDS else ""

!python scripts/train.py \
  --model configs/model/viora_pragmatic.yaml \
  --train configs/training/pragmatic_lora.yaml \
  --shards "{SHARDS}" {_val_shards_flag} --qa-prob 0.7 \
  llm.name_or_path=Qwen/Qwen2.5-0.5B-Instruct \
  training.precision=fp16 training.batch_size=4 training.num_workers=4 \
  training.gradient_checkpointing=true training.keep_last_checkpoints=2 {_resume_flag} \
  training.max_steps={TARGET_STEPS_THIS_SESSION} training.save_every=200 training.log_every=20 \
  {_eval_every_override} \
  training.output_dir={OUT}


In [ ]:
# 6b) Push this session's checkpoint to HF so the NEXT session can resume from it.
#      Pushes whichever checkpoint was actually written MOST RECENTLY -- NOT hardcoded
#      "final.pt". A session that stops before reaching its step target (crash, time
#      limit, manual stop -- the COMMON case, not the exception) never rewrites
#      final.pt; the real progress is in the newest step_*.pt. Always stored on HF
#      under the name "final.pt" so cell 4b's pull (which always requests that name)
#      finds it next session.
from viora.training.checkpointing import latest_checkpoint
from viora.utils.hf_relay import push_checkpoint_to_hf

ckpt_to_push = latest_checkpoint(OUT)
assert ckpt_to_push is not None, f"no checkpoint found in {OUT} to push"
print(f"pushing {ckpt_to_push.name} (most recently written checkpoint)...")
url = push_checkpoint_to_hf(str(ckpt_to_push), HF_REPO, path_in_repo="final.pt", token=HF_TOKEN)
print(f"pushed -> {url}  (next session will auto-resume from this)")


In [ ]:
# 7) See what it learned: caption real clips with your trained checkpoint.
#    Works on ANY checkpoint (a step_*.pt mid-run, or final.pt) -- no need to wait for the end.
#    Picks whichever checkpoint was actually written MOST RECENTLY, not just "final.pt" by
#    name -- final.pt is only (re)written when training completes a full run; a session that
#    trains further but stops early (crash/time limit) leaves newer step_*.pt sitting next to
#    a now-STALE final.pt, and preferring the name over the timestamp would silently show old
#    results.
import glob, os, torch
from viora.utils.config import load_config
from viora.models.viora import VioraForVideoUnderstanding
from viora.inference.pipeline import VioraInferencePipeline
from viora.training.checkpointing import latest_checkpoint, load_checkpoint

OUT = OUT if "OUT" in globals() else "/kaggle/working/pragmatic"
INPUT_ROOT = INPUT_ROOT if "INPUT_ROOT" in globals() else "/kaggle/input"

CKPT = latest_checkpoint(OUT)
assert CKPT is not None, f"no checkpoint in {OUT} yet — let training reach its first save (step 200)."
print("checkpoint:", CKPT)

# rebuild the SAME model as training, then load the trained weights
mcfg = load_config("configs/model/viora_pragmatic.yaml",
                   overrides=["llm.name_or_path=Qwen/Qwen2.5-0.5B-Instruct"])
model = VioraForVideoUnderstanding(mcfg)
load_checkpoint(CKPT, model)
dev = "cuda" if torch.cuda.is_available() else "cpu"
pipe = VioraInferencePipeline(model, device=dev)

# caption a few real clips
for vp in glob.glob(f"{INPUT_ROOT}/**/*.mp4", recursive=True)[:5]:
    cap, conf = pipe.caption(pipe.index(vp))
    print(f"\n{os.path.basename(vp)}\n  -> {cap!r}  (conf {conf:.2f})")

In [ ]:
# 7b) Ask a question about a real clip (uses the SAME checkpoint loaded in cell 7).
QUESTION = "what is happening in this video"

video_path = glob.glob(f"{INPUT_ROOT}/**/*.mp4", recursive=True)[0]
idx = pipe.index(video_path)
answer, conf = pipe.generate_answer(idx, QUESTION, model.llm.tokenizer)
print(f"{os.path.basename(video_path)}\nQ: {QUESTION}\nA: {answer!r}  (conf {conf:.2f})")


## Resuming after a session times out

Re-run cells 1–5 (and 5b if using real data), then add `training.resume=<checkpoint>` to cell 6 — e.g.:

```
  training.resume={OUT}/step_2000.pt
```

It restores model + optimizer + step and continues. Repeat across free sessions until done —
this is how you reach 20k–50k steps on a free GPU. Old `step_*.pt` are auto-pruned (newest 3 kept)
so they don't fill Kaggle's 20 GB disk.

## Full-fledged (real) dataset — MSR-VTT

Cell **5b** turns a real dataset into Viora shards. On Kaggle: right panel → **Add Data** → search
**MSR-VTT** → add a version with the `.mp4` clips + a `*videodatainfo.json*`, then run cell 5b (it
auto-finds them). ~6.5k train clips × ~20 captions each = ~130k caption views.

**Any other dataset (MSVD, your own videos):** make a JSON sidecar `{ "clip1": "a caption",
"clip2": ["cap a", "cap b"] }` and run:

```
python scripts/prepare_video_dataset.py --videos <dir> --annotations captions.json \
    --format folder --out data/shards/train-%06d.tar
```

### What this training gives you (honest scope)

Cell **5c** builds a COMBINED caption+MSRVTT-QA shard set from the SAME videos (no extra
download): each view is randomly rendered as a caption or a `Question:/Answer:` pair
(`--qa-prob`, default 0.5), so the model learns to both **describe** clips and **answer
questions** about them. Cells **4b**/**6b** relay the latest checkpoint through your HF repo
(`HF_REPO`) so a multi-session run (~20k total steps) resumes automatically — no re-download,
no restart from step 0. If cell 5c can't find a working MSRVTT-QA source, training falls back to
caption-only (cell 5b's shards) automatically.

The **evidence timestamps** are still not meaningful — that needs a **temporal-grounding** set
(e.g. Charades-STA), which is out of scope for this run. Bigger LLM
(`Qwen/Qwen2.5-1.5B-Instruct`) + more steps + more data = better quality still.

## Serve it

```
VIORA_MODEL_CONFIG=configs/model/viora_pragmatic.yaml VIORA_CHECKPOINT={OUT}/final.pt \
  uvicorn viora.serving.api:app --host 0.0.0.0 --port 8000
```